# Backfill Tuner Columns

This notebook backfills three columns in `prod.detection.viewing_content_golden`:
- **tuner_series_id** - from `epg_show.series_id` (TIVO vendor)
- **tuner_tms_epi_title** - from `epg_show.epi_title` (TMS vendor)
- **tuner_tivo_epi_title** - from `epg_show.epi_title` (TIVO vendor)


In [0]:
# Generate list of dates since 
# Use this to create parallel job runs
from datetime import datetime, timedelta

end_date = datetime.now().date()
start_date = end_date - timedelta(days=1)
end_date = end_date - timedelta(days=1)

date_list = []
current = start_date
while current <= end_date:
    date_list.append(current.strftime('%Y-%m-%d'))
    current += timedelta(days=1)

print(f"Generated {len(date_list)} dates from {start_date} to {end_date}")

In [0]:
# # Spark optimizations for large MERGE operations
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.databricks.photon.sort.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.databricks.adaptive.autoOptimizeShuffle.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

In [0]:
for target_date in date_list:
    spark.sql(f"""
    MERGE INTO prod.detection_onn.viewing_content_golden AS target
    USING (
        SELECT 
            c.fk_tvid,
            c.session_start,
            c.session_end,
            SUBSTRING(tuner_tivo_show.series_id, 1, 256) AS tuner_series_id,
            SUBSTRING(NULLIF(REGEXP_REPLACE(tuner_tms_show.epi_title, '[\“\”\"\^\@\,]', ''), ''), 1, 256) AS tuner_tms_epi_title,
            SUBSTRING(NULLIF(REGEXP_REPLACE(tuner_tivo_show.epi_title, '[\“\”\"\^\@\,]', ''), ''), 1, 256) AS tuner_tivo_epi_title,
            c.tuner_channel_number, 
            CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id, 
            CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id, 
            REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tms_title, 
            REGEXP_REPLACE(CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.title END, '[\“\”\"\^\@\,]', '') AS tuner_tivo_title, 
            CASE WHEN c.file_ingested THEN NULL ELSE c.tms_tuner_airdate END AS tuner_tms_airdate, 
            CASE WHEN c.file_ingested THEN NULL ELSE c.tuner_airdate     END AS tuner_tivo_airdate
        FROM prod.detection_onn.viewing_content_firehose AS c
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tivo_show
            ON tuner_tivo_show.show_id = c.tuner_program_id
            AND tuner_tivo_show.vendor_name = 'TIVO'
        LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
            ON tuner_tms_show.show_id = c.tms_tuner_program_id
            AND tuner_tms_show.vendor_name = 'TMS'
        WHERE c.partition_key = '{target_date}'
        AND COALESCE(c.tuner_program_id, c.tms_tuner_program_id) IS NOT NULL
        GROUP BY ALL
    ) AS source
    ON target.fk_tvid = source.fk_tvid
    AND target.session_start = source.session_start
    AND target.session_end = source.session_end
    AND target.tuner_channel_number <=> source.tuner_channel_number
    AND target.tuner_tms_episode_id <=> source.tuner_tms_episode_id
    AND target.tuner_tivo_episode_id <=> source.tuner_tivo_episode_id
    AND target.tuner_tms_title <=> source.tuner_tms_title
    AND target.tuner_tivo_title <=> source.tuner_tivo_title
    AND target.tuner_tms_airdate <=> source.tuner_tms_airdate
    AND target.tuner_tivo_airdate <=> source.tuner_tivo_airdate
    AND target.session_start_hour >= '{target_date}'
    AND target.session_start_hour < DATE_ADD('{target_date}', 1)
    AND COALESCE(target.tuner_tms_title, target.tuner_tivo_title, target.tuner_tms_episode_id, target.tuner_tivo_episode_id) IS NOT NULL
    AND COALESCE(target.tuner_series_id, target.tuner_tms_epi_title, target.tuner_tivo_epi_title) IS NULL
    WHEN MATCHED THEN UPDATE SET
        target.tuner_series_id = source.tuner_series_id,
        target.tuner_tms_epi_title = source.tuner_tms_epi_title,
        target.tuner_tivo_epi_title = source.tuner_tivo_epi_title;
    """)
    print(f'finished {target_date}')

In [0]:
%sql
SELECT token, tv_count FROM (
SELECT token, COUNT(DISTINCT tvid) AS tv_count FROM prod.detection.tv
WHERE joined_date >= CURRENT_DATE - 365
GROUP BY 1)
WHERE tv_count > 4
AND tv_count < 100
ORDER BY tv_count DESC
LIMIT 100

In [0]:
%sql
SELECT * FROM prod.detection.tv
WHERE token IN ('3c754c7cae5805fde37a5f02c9602b28', 'cad61b1dd21eb6ec0ca5884f36aa0657', '34cce50102a9ae2df2abf118d6c480ae', '5f3bfdeba2fe85e9c8e8208a6896fd27')
ORDER BY token, joined_date DESC

In [0]:
%sql
SELECT token, tv_ad_id, tvid FROM prod.detection.tv_ad_id
WHERE token IN ('3c754c7cae5805fde37a5f02c9602b28', 'cad61b1dd21eb6ec0ca5884f36aa0657', '34cce50102a9ae2df2abf118d6c480ae', '5f3bfdeba2fe85e9c8e8208a6896fd27')
GROUP BY token, tv_ad_id, tvid
ORDER BY token, tv_ad_id, tvid

In [0]:
# Summary: total hours to fix and most common violated columns
violations_df = spark.table("dev.public.viewing_content_golden_char_violations_new")
print(f"Total hours with violations: {violations_df.count()}")
display(violations_df)